In [1]:
import os
import json
from tqdm.auto import tqdm
from pathlib import Path

import pandas as pd

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# RUN ONCE: change working directory to project root
cwd = Path.cwd()

pwd = cwd.parent

os.chdir(pwd)
print(f"Changed working directory to {pwd}")

if Path.cwd() != pwd:
    raise RuntimeError(f"Failed to change working directory to {pwd}")

Changed working directory to /Users/jdk/projects/recruiting-reader


In [3]:
# # create driver
# from src.scraper.driver import make_driver
# driver = make_driver(headless=True)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def make_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,1000")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    # helps reduce some bot friction
    opts.add_argument("--lang=en-US")
    driver = webdriver.Chrome(options=opts)  # Selenium Manager will fetch driver if needed
    driver.set_page_load_timeout(45)
    return driver

driver = make_driver(headless=True)

In [ ]:
# driver.quit()

### Investigate Missing Values

In [4]:
with open("data/urls_2025_transfers.json", "r") as f:
    urls_2025 = json.load(f)

In [6]:
portal_df_test = pd.read_csv("data/portal_2025_transfers_1218_run5.csv")

In [7]:
portal_df_test.isna().sum()

id_247                     0
name                       0
pos_247                    4
hs_name                   11
hs_city                    6
hs_state                   6
transfer_rating            6
transfer_year              0
transfer_ovr_rank         93
transfer_pos_rank         41
transfer_stars             6
transfer_origin            2
transfer_destination     134
hs_class                   5
hs_rating_247           1012
hs_pos                     4
composite_rating        1121
composite_natl_rank     1123
composite_pos_rank      1123
source_hs_url              4
hs_stars                1012
source_player_url          0
transfer_status         2867
dtype: int64

In [23]:
# display(portal_df_test[portal_df_test['transfer_destination'].isna()].head())
# print(portal_df_test[portal_df_test['transfer_destination'].isna()].iloc[2]['source_player_url'])
# # 46137152 dropped out

display(portal_df_test[portal_df_test['hs_name'].isna()].head(11))
print(portal_df_test[portal_df_test['hs_name'].isna()].iloc[-2]['source_player_url'])
print(portal_df_test[portal_df_test['hs_name'].isna()].iloc[-2])
# https://247sports.com/player/louis-brown-iv-46111905/college-310862 is a problem
# https://247sports.com/player/easton-messer-46103582/college-286927/

# display(portal_df_test[portal_df_test['transfer_rating'].isna()].head(11))
# print(portal_df_test[portal_df_test['transfer_rating'].isna()].iloc[2]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_rating'].isna()].iloc[2])
# fine

# display(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].head(11))
# print(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].iloc[4]['source_player_url'])
# print(portal_df_test[portal_df_test['transfer_ovr_rank'].isna() &
#                        portal_df_test['transfer_rating'].notna()].iloc[4])
# fine




,id_247,name,pos_247,hs_name,hs_city,hs_state,transfer_rating,transfer_year,transfer_ovr_rank,transfer_pos_rank,...,hs_class,hs_rating_247,hs_pos,composite_rating,composite_natl_rank,composite_pos_rank,source_hs_url,hs_stars,source_player_url,transfer_status
120,46042923,This site can’t be reached,NaN,NaN,NaN,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://247sports.com/player/joshua-eaton-4604...,NaN
678,46128460,Nuer Gatkuoth,EDGE,NaN,Edmonton,AB,86.0,2025,712.0,69.0,...,2022.0,NaN,LB,NaN,NaN,NaN,https://247sports.com/player/nuer-gatkuoth-461...,NaN,https://247sports.com/player/nuer-gatkuoth-461...,NaN
680,46137816,Melvin Siani,OT,NaN,Ontario,CA,86.0,2025,717.0,64.0,...,2023.0,NaN,OT,NaN,NaN,NaN,https://247sports.com/player/melvin-siani-4613...,NaN,https://247sports.com/player/melvin-siani-4613...,NaN
859,46056791,Your connection was interrupted,NaN,NaN,NaN,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://247sports.com/player/dekel-crowdus-460...,NaN
1610,46103582,Easton Messer,WR,NaN,NaN,NaN,NaN,2025,NaN,NaN,...,2022.0,77.0,WR,NaN,NaN,NaN,https://247sports.com/player/easton-messer-461...,2.0,https://247sports.com/player/easton-messer-461...,NaN
1737,46140445,Vili Taufatofua,EDGE,NaN,New Zealand,NEW,83.0,2025,2127.0,197.0,...,2023.0,NaN,DL,NaN,NaN,NaN,https://247sports.com/player/vili-taufatofua-4...,NaN,https://247sports.com/player/vili-taufatofua-4...,NaN
2278,46158069,Kenton Allen,LB,NaN,Riverside,CA,83.0,2025,2256.0,176.0,...,2023.0,NaN,LB,NaN,NaN,NaN,https://247sports.com/player/kenton-allen-4615...,NaN,https://247sports.com/player/kenton-allen-4615...,NaN
2330,46141053,Charlie Leota,DL,NaN,Auckland,AC,82.0,2025,2315.0,246.0,...,2023.0,NaN,DL,NaN,NaN,NaN,https://247sports.com/player/charlie-leota-461...,NaN,https://247sports.com/player/charlie-leota-461...,NaN
2679,46155126,This site can’t be reached,NaN,NaN,NaN,NaN,NaN,2025,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://247sports.com/player/simon-mapa-461551...,NaN
2842,46111905,Louis Brown IV,WR,NaN,NaN,NaN,NaN,2025,NaN,NaN,...,2022.0,84.0,WR,0.8207,1882.0,233.0,https://247sports.com/player/louis-brown-iv-46...,3.0,https://247sports.com/player/louis-brown-iv-46...,NaN


https://247sports.com/player/louis-brown-iv-46111905/college-310862
id_247                                                           46111905
name                                                       Louis Brown IV
pos_247                                                                WR
hs_name                                                               NaN
hs_city                                                               NaN
hs_state                                                              NaN
transfer_rating                                                       NaN
transfer_year                                                        2025
transfer_ovr_rank                                                     NaN
transfer_pos_rank                                                     NaN
transfer_stars                                                        NaN
transfer_origin                                            Colorado State
transfer_destination                        

In [4]:
from src.utils.tests import test_timeline
test_timeline(driver,
              'https://247sports.com/player/markevious-brown-46052018/college-300599/')

2025-04-08 Transfer | Markevious Brown entered the transfer portal
2023-06-22 Transfer | Markevious Brown transfers to Purdue Boilermakers
2023-04-19 Transfer | Markevious Brown entered the transfer portal
2021-01-01 Enrolled | Markevious Brown enrolls at Ole Miss Rebels
2021-01-01 O. Visit | Markevious Brown officially visits Ole Miss Rebels


[TL(date=datetime.datetime(2025, 4, 8, 0, 0), kind='Transfer', text='Markevious Brown entered the transfer portal'),
 TL(date=datetime.datetime(2023, 6, 22, 0, 0), kind='Transfer', text='Markevious Brown transfers to Purdue Boilermakers'),
 TL(date=datetime.datetime(2023, 4, 19, 0, 0), kind='Transfer', text='Markevious Brown entered the transfer portal'),
 TL(date=datetime.datetime(2021, 1, 1, 0, 0), kind='Enrolled', text='Markevious Brown enrolls at Ole Miss Rebels'),
 TL(date=datetime.datetime(2021, 1, 1, 0, 0), kind='O. Visit', text='Markevious Brown officially visits Ole Miss Rebels')]

In [9]:
# import importlib

# import src.scraper.player_scraper 
# importlib.reload(src.scraper.player_scraper)

In [24]:
from src.scraper.player_scraper import scrape_player

p = scrape_player(driver,
                  'https://247sports.com/player/easton-messer-46103582/college-286927/')
p
# # https://247sports.com/player/louis-brown-iv-46111905/college-310862 is a problem
# # https://247sports.com/player/easton-messer-46103582/college-286927/


TIMELINE CHECK for Easton Messer
2024-12-23 Transfer | Easton Messer commits to Florida Atlantic Owls
2024-12-09 Transfer | Easton Messer entered the transfer portal
2022-07-01 Enrolled | Easton Messer enrolls at Western Kentucky...
2022-02-02 Signed | Easton Messer signs letter of intent to Western...
2022-01-15 Commit | Easton Messer commits to Western Kentucky...
INSTITUTION CANDIDATES: ['Florida Atlantic', 'Western Kentucky']
Easton Messer commits to Florida Atlantic Owls
<re.Match object; span=(14, 24), match='commits to'>
Easton Messer entered the transfer portal
None
Easton Messer enrolls at Western Kentucky...
<re.Match object; span=(14, 24), match='enrolls at'>


{'id_247': 46103582,
 'name': 'Easton Messer',
 'pos_247': 'WR',
 'hs_name': 'Christian Academy of Louisville',
 'hs_city': 'Louisville',
 'hs_state': 'KY',
 'transfer_rating': 84,
 'transfer_year': 2025,
 'transfer_ovr_rank': 1663,
 'transfer_pos_rank': 264,
 'transfer_stars': 3,
 'transfer_origin': 'Western Kentucky',
 'transfer_destination': 'Florida Atlantic',
 'transfer_status': 'committed',
 'hs_class': 2022,
 'hs_rating_247': 77,
 'hs_pos': 'WR',
 'composite_rating': None,
 'composite_natl_rank': None,
 'composite_pos_rank': None,
 'source_hs_url': 'https://247sports.com/player/easton-messer-46103582/high-school-254667',
 'hs_stars': 2,
 'source_player_url': 'https://247sports.com/player/easton-messer-46103582/college-286927/'}

In [9]:
from src.storage.cache_new import load_cache

CACHE_PATH = "data/portal_2025_transfers_1218_run5.jsonl"

test = load_cache(CACHE_PATH)

In [11]:
test_clean = {
    k: v
    for k, v in test.items()
    if (
        isinstance(v, dict)
        and "name" in v
        and v.get("transfer_origin") is not None
        and v.get("transfer_destination") is not None
    )
}

In [14]:
3008 - len(test_clean)

142